# Evaluate web search tool call accuracy

This notebook uses the Azure AI Evaluation SDK to score whether each response invoked the web search tool. For now, the evaluator simply checks if any tool call was made. If your dataset includes `expected_tool_call`, the summary will also compute accuracy.

In [ ]:
# Optional setup: install the Azure AI Evaluation SDK in the notebook kernel.
# %pip install azure-ai-evaluation

In [12]:
# Import core utilities, typing helpers, and the evaluation entry point.
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, TypedDict

from azure.ai.evaluation import evaluate

try:
    import pandas as pd
except ImportError:
    pd = None

In [13]:
# Define helpers to load JSONL files and extract tool calls from Responses API payloads.
TOOL_CALL_TYPES = {"tool_call", "web_search_call"}


def load_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def extract_tool_calls(output_items: list[dict]) -> list[dict]:
    tool_calls: list[dict] = []
    for item in output_items or []:
        if item.get("type") in TOOL_CALL_TYPES:
            tool_calls.append(item)
        for content_item in item.get("content", []) or []:
            if content_item.get("type") in TOOL_CALL_TYPES:
                tool_calls.append(content_item)
    return tool_calls

In [14]:
# Build a lightweight evaluator that scores tool call presence (1.0 = tool call).
class EvalOutput(TypedDict, total=False):
    result: float
    tool_call_made: bool
    tool_call_correct: Optional[bool]


def tool_call_evaluator(
    query: Optional[str] = None,
    response: Optional[Dict[str, Any]] = None,
    error: Optional[Any] = None,
    expected_tool_call: Optional[bool] = None,
) -> EvalOutput:
    tool_call_made = False
    if isinstance(response, dict):
        tool_calls = extract_tool_calls(response.get("output", []))
        tool_call_made = len(tool_calls) > 0
    elif error is not None:
        tool_call_made = False

    tool_call_correct = (
        tool_call_made == expected_tool_call
        if expected_tool_call is not None
        else None
    )
    return {
        "result": 1.0 if tool_call_made else 0.0,
        "tool_call_made": tool_call_made,
        "tool_call_correct": tool_call_correct,
    }


def aggregate_tool_call_metrics(rows: List[Dict[str, Any]]) -> Dict[str, float]:
    scores = [
        row.get("result")
        for row in rows
        if isinstance(row, dict) and row.get("result") is not None
    ]
    total = len(scores)
    tool_call_count = sum(1 for score in scores if score > 0)
    metrics = {
        "tool_call_count": float(tool_call_count),
        "tool_call_rate": tool_call_count / total if total else 0.0,
    }
    correct_flags = [
        row.get("tool_call_correct")
        for row in rows
        if row.get("tool_call_correct") is not None
    ]
    if correct_flags:
        metrics["tool_call_accuracy"] = sum(1 for flag in correct_flags if flag) / len(
            correct_flags
        )
    return metrics


tool_call_evaluator.__aggregate__ = aggregate_tool_call_metrics

In [ ]:
# Load the latest raw response log and merge in expected tool call labels.
output_dir = Path("../outputs")
jsonl_paths = sorted(output_dir.glob("web_search_eval_raw_*.jsonl"))
if not jsonl_paths:
    raise FileNotFoundError(
        "Run generate_eval_dataset.ipynb to create a web_search_eval_raw_*.jsonl file."
    )
data_path = jsonl_paths[-1]
records = load_jsonl(data_path)

expected_path = Path("../data/search_eval.jsonl")
if expected_path.exists():
    expected_records = load_jsonl(expected_path)
    expected_by_id = {
        record.get("id"): record.get("expected_tool_call")
        for record in expected_records
        if record.get("id") is not None
    }
    for record in records:
        record.setdefault("expected_tool_call", expected_by_id.get(record.get("id")))

merged_path = output_dir / f"{data_path.stem}_with_expected.jsonl"
payload = "\n".join(json.dumps(record, ensure_ascii=False) for record in records) + "\n"
merged_path.write_text(payload, encoding="utf-8")

lines = merged_path.read_text(encoding="utf-8").splitlines()
print(f"Using data file: {merged_path.resolve()}")
print(f"Total lines: {len(lines)}")
print("Preview (first 3 lines):")
print("\n".join(lines[:3]))

Using data file: C:\repo\ai-foundry-craftkit\Model_Usecases\web-search\evals\outputs\web_search_eval_raw_20260309_083517_with_expected.jsonl
Total lines: 8
Preview (first 3 lines):
{"id": "search_001", "query": "Latest NASA Artemis mission update", "test_case_description": "Needs recent information; should use web search.", "response": {"id": "resp_0cd58d66e07839b40069aedf8e1b28819594bd8c9bb792e662", "created_at": 1773068174.0, "error": null, "incomplete_details": null, "instructions": null, "metadata": {}, "model": "gpt-5.2", "object": "response", "output": [{"id": "ws_0cd58d66e07839b40069aedf8ed67881958eef68cc0270229d", "action": {"query": "NASA Artemis latest mission update March 2026", "type": "search", "queries": ["NASA Artemis latest mission update March 2026", "NASA Artemis II schedule update 2026", "NASA Artemis III schedule update", "NASA Artemis IV update"], "sources": null}, "status": "completed", "type": "web_search_call"}, {"id": "msg_0cd58d66e07839b40069aedf92762c819595f8

In [16]:
# Run the evaluator with the Azure AI Evaluation SDK and print summary metrics.
eval_output = evaluate(
    data=merged_path,
    evaluators={"tool_call": tool_call_evaluator},
    _use_pf_client=False,
)

row_results = [row["outputs.tool_call.result"] for row in eval_output["rows"]]
metrics = eval_output["metrics"]

print(f"tool call results: {row_results}")
print("aggregated metrics:")
print(json.dumps(metrics, indent=2))
metrics

2026-03-09 11:22:12 -0700   59508 execution.bulk     INFO     Finished 8 / 8 lines.
2026-03-09 11:22:12 -0700   59508 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.
======= Run Summary =======

Run name: "tool_call_20260309_182212_349470"
Run status: "Completed"
Start time: "2026-03-09 18:22:12.349470+00:00"
Duration: "0:00:01.015820"

======= Combined Run Summary (Per Evaluator) =======

{
    "tool_call": {
        "status": "Completed",
        "duration": "0:00:01.015820",
        "completed_lines": 8,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    }
}


tool call results: [1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
aggregated metrics:
{
  "tool_call.result": 0.875,
  "tool_call.tool_call_made": 0.875,
  "tool_call.tool_call_correct": 1.0,
  "tool_call.tool_call_count": 7.0,
  "tool_call.tool_call_rat

{'tool_call.result': 0.875,
 'tool_call.tool_call_made': 0.875,
 'tool_call.tool_call_correct': 1.0,
 'tool_call.tool_call_count': 7.0,
 'tool_call.tool_call_rate': 0.875,
 'tool_call.tool_call_accuracy': 1.0}

In [17]:
# Inspect the first few evaluation rows in a DataFrame (if pandas is available).
if pd:
    df = pd.DataFrame(eval_output["rows"])
    df.head()
else:
    eval_output["rows"][:5]

In [18]:
# List queries that did not trigger a tool call for manual review.
missing_tool_calls = [
    row.get("inputs.query")
    for row in eval_output["rows"]
    if not row.get("outputs.tool_call.tool_call_made")
]
missing_tool_calls[:10]

['Define photosynthesis in one sentence.']

In [ ]:
# Placeholder cell for additional evaluation ideas or notes.
